In [1]:
import os
import glob
import re
import cv2
import pandas as pd
import pytesseract

In [2]:
def categorize_item(item_text):
    categories = {
        "office_supplies": ["paper", "pen", "notebook", "staple", "folder"],
        "electronics": ["laptop", "monitor", "mouse", "keyboard", "printer"],
        "services": ["consult", "service", "maintenance", "support"],
        "shipping": ["delivery", "shipping", "freight", "courier"],
        "software": ["license", "subscription", "software", "saas"],
    }
    lowered = item_text.lower()
    for category, keywords in categories.items():
        if any(keyword in lowered for keyword in keywords):
            return category
    return "other"

In [3]:
def extract_items_from_text(text):
    if "ITEMS" in text:
        items_section = text.split("ITEMS", 1)[1]
    else:
        items_section = text

    # Stop at common totals markers if present
    items_section = re.split(r"\n\s*(TOTAL|SUBTOTAL|AMOUNT DUE)\b", items_section, maxsplit=1)[0]

    lines = [line.strip() for line in items_section.splitlines() if line.strip()]
    items = []
    for line in lines:
        if re.match(r"^\d+\.\s+", line):
            items.append(line)

    return items

In [4]:
def extract_invoice_items(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Failed to load image: {image_path}")
        return []

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(gray)

    invoice_no = re.search(r"Invoice no:\s*(\d+)", text)
    invoice_no = invoice_no.group(1) if invoice_no else ""

    items = extract_items_from_text(text)
    rows = []
    for item_line in items:
        description = re.sub(r"^\d+\.\s+", "", item_line).strip()
        category = categorize_item(description)
        rows.append({
            "invoice_id": invoice_no,
            "item_description": description,
            "category": category,
        })

    return rows

In [5]:
invoice_folder = "invoices"
image_files = glob.glob(os.path.join(invoice_folder, "*.jpg"))

all_rows = []


In [6]:
for image_file in image_files:
    all_rows.extend(extract_invoice_items(image_file))

In [7]:
items_df = pd.DataFrame(all_rows)
items_df.to_csv("invoice_items_with_categories.csv", index=False)
print("Saved invoice_items_with_categories.csv")

Saved invoice_items_with_categories.csv
